In [1]:
import json

from src.utils import parse_orig_sql
from datasets import load_dataset

with open('../../data/spider/tables.json', 'r', encoding='utf-8') as f:
    spider_tables = json.load(f)

spider_train = load_dataset("xlangai/spider", split="train")

spider_val = load_dataset("xlangai/spider", split="validation")


D:\Master\Grundprojekt\Grundprojekt_Schema_Linking_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from collections import defaultdict
import re

def find_query_multiple_tables(dataset):
    query_multiple_tables = []

    for q in dataset:
        gold_schema = parse_orig_sql(q['query'])
        if len(gold_schema.keys()) > 1:
            query_multiple_tables.append({"query": q['query'], "gold_schema": gold_schema, "db_id": q['db_id']})

    return query_multiple_tables

def tables_connected_by_fk_spider(db_id, table_names):
    """Prueft, ob alle gegebenen Tabellen im FK-Graph der DB zusammenhaengen
    (auch transitiv ueber Zwischentabellen, die nicht im gold_schema stehen)."""
    db = next(db for db in spider_tables if db['db_id'] == db_id)
    orig_tables = db['table_names_original']
    col_names = db['column_names_original']

    name_to_idx = {name.lower(): idx for idx, name in enumerate(orig_tables)}
    target_idxs = [name_to_idx[t.lower()] for t in table_names if t.lower() in name_to_idx]

    if len(target_idxs) < 2:
        return False

    adjacency = {i: set() for i in range(len(orig_tables))}
    for fk_from, fk_to in db['foreign_keys']:
        t_from = col_names[fk_from][0]
        t_to = col_names[fk_to][0]
        adjacency[t_from].add(t_to)
        adjacency[t_to].add(t_from)

    visited = set()
    stack = [target_idxs[0]]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        stack.extend(adjacency[node] - visited)

    return all(idx in visited for idx in target_idxs)

def find_query_tables_fk_connection(dataset):
    query_multiple_tables = find_query_multiple_tables(dataset)
    query_multiple_tables_fk_connected = [
        q for q in query_multiple_tables
        if tables_connected_by_fk_spider(q['db_id'], q['gold_schema'].keys())
    ]

    query_multiple_tables_not_fk_connected = [
        q for q in query_multiple_tables
        if not tables_connected_by_fk_spider(q['db_id'], q['gold_schema'].keys())
    ]

    print(f"verbunden: {len(query_multiple_tables_fk_connected)} / insgesamt: {len(query_multiple_tables)}")
    return query_multiple_tables_fk_connected, query_multiple_tables_not_fk_connected


# --- Spider-Ent: die FKs stehen hier nicht in einer eigenen foreign_keys-Liste,
# sondern als "FOREIGN KEY (...) REFERENCES `table` (...)" direkt im DDL-String
# -> per Regex rausziehen, statt ueber Spalten-Indizes wie bei Spider.
_FK_REF_REGEX = re.compile(r'FOREIGN KEY\s*\([^)]*\)\s*REFERENCES\s*`?([^`\s(]+)`?\s*\(', re.IGNORECASE)

def get_ent_fk_edges(data_asset):
    """Baut die FK-Kanten (Tabelle <-> Tabelle) fuer einen Spider-Ent data_asset aus den DDLs."""
    edges = []
    for table_info in schema_with_parsed_candidates.get(data_asset, []):
        table_name = table_info['candidates']['table']
        if not table_name:
            continue
        for ref_table in _FK_REF_REGEX.findall(table_info['ddl']):
            edges.append((table_name.lower(), ref_table.lower()))
    return edges

def tables_connected_by_fk_ent(data_asset, table_names):
    """Prueft, ob alle gegebenen Tabellen im FK-Graph des Spider-Ent data_assets
    zusammenhaengen (auch transitiv ueber Zwischentabellen, die nicht im gold_schema stehen)."""
    target = {t.lower() for t in table_names if t}

    if len(target) < 2:
        return False

    adjacency = defaultdict(set)
    for t_from, t_to in get_ent_fk_edges(data_asset):
        adjacency[t_from].add(t_to)
        adjacency[t_to].add(t_from)

    start = next(iter(target))
    visited = set()
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        stack.extend(adjacency[node] - visited)

    return all(t in visited for t in target)

def find_query_multiple_tables_ent(dataset):
    query_multiple_tables = []

    for q in dataset:
        gold_schema = get_ent_gold_schema(q)
        if len(gold_schema.keys()) > 1:
            query_multiple_tables.append({
                "query": q['original_SQL'],
                "gold_schema": gold_schema,
                "data_asset": q['data_asset'],
            })

    return query_multiple_tables

def find_query_tables_fk_connection_ent(dataset):
    query_multiple_tables = find_query_multiple_tables_ent(dataset)
    query_multiple_tables_fk_connected = [
        q for q in query_multiple_tables
        if tables_connected_by_fk_ent(q['data_asset'], q['gold_schema'].keys())
    ]

    query_multiple_tables_not_fk_connected = [
        q for q in query_multiple_tables
        if not tables_connected_by_fk_ent(q['data_asset'], q['gold_schema'].keys())
    ]

    print(f"verbunden: {len(query_multiple_tables_fk_connected)} / insgesamt: {len(query_multiple_tables)}")
    return query_multiple_tables_fk_connected, query_multiple_tables_not_fk_connected


In [ ]:
print("spider_train")
find_query_tables_fk_connection(spider_train)

print("spider_val")
find_query_tables_fk_connection(spider_val)

In [ ]:
import re

# Prueft, wie oft in einer Query ein Alias mit anderem Case referenziert wird, als er
# in "AS <alias>" definiert wurde (z.B. "JOIN ship AS t2" ... "T2.id"). Das ist genau der
# Fall, an dem die frueher case-sensitive Alias-Aufloesung in utils.py._handle_select
# gescheitert ist (siehe Fix: alias_map.get(alias.lower(), alias)).
#
# Bewusst regex-basiert statt ueber mo_sql_parsing, um komplett unabhaengig vom (jetzt
# gefixten) Code in utils.py zu sein - es soll nur der reine SQL-Text auf das Muster
# geprueft werden, nicht die Aufloesung selbst.
_ALIAS_DEF_REGEX = re.compile(r'\b(?:FROM|JOIN)\s+`?(\w+)`?\s+AS\s+(\w+)\b', re.IGNORECASE)
_ALIAS_REF_REGEX = re.compile(r'\b(\w+)\.\w+\b')

def find_alias_case_mismatch(query: str):
    """Gibt die Liste der Aliase zurueck, die in `query` mit anderem Case referenziert
    werden, als sie definiert wurden (case-insensitiv gleich, aber nicht case-sensitiv)."""
    alias_defs = {}  # alias wie geschrieben -> table
    for table, alias in _ALIAS_DEF_REGEX.findall(query):
        alias_defs[alias] = table
    alias_defs_lower = {a.lower(): a for a in alias_defs}

    mismatches = set()
    for ref_alias in _ALIAS_REF_REGEX.findall(query):
        if ref_alias in alias_defs:
            continue  # exakter Case-Match, kein Problem
        if ref_alias.lower() in alias_defs_lower:
            mismatches.add(ref_alias)
    return sorted(mismatches)


def count_alias_case_mismatches(dataset, name: str):
    affected = []
    for q in dataset:
        mismatches = find_alias_case_mismatch(q['query'])
        if mismatches:
            affected.append({"query": q['query'], "db_id": q['db_id'], "mismatched_aliases": mismatches})

    print(f"{name}: {len(affected)} / {len(dataset)} Queries mit Alias-Case-Mismatch "
          f"({len(affected) / len(dataset):.2%})")
    return affected


mismatches_train = count_alias_case_mismatches(spider_train, "spider_train")
mismatches_val = count_alias_case_mismatches(spider_val, "spider_val")


In [2]:
from src.data_loaders.spider_ent_data import get_spider_ent_data, get_ent_gold_schema, schema_with_parsed_candidates
from src.data_loaders.spider_ent_data import spider_ent as spider_ent_raw

spider_ent = get_spider_ent_data()
print()

KeyboardInterrupt: 

In [ ]:
print("spider_ent")
find_query_tables_fk_connection_ent(spider_ent_raw)
